<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-openai \
    tavily-python

## Without web search

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [ ]:
import os
import time
from langchain import chat_models, agents, messages

# Initialize using the OpenAI provider with a custom base_url
llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

agent = agents.create_agent(model=llm)

question = messages.HumanMessage(content="""
    How up to date is your training knowledge?
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 2.11s seconds
================================ Human Message =================================


    How up to date is your training knowledge?

================================== Ai Message ==================================

My training data includes information that was collected up until **June 2024**.  
Anything that happened or was published after that point isn’t part of my built‑in knowledge, and I don’t have the ability to browse the web in real time.  

If you need the very latest facts, figures, or developments (e.g., breaking news, recent scientific papers, newly released software versions, etc.), I recommend checking:

- Reputable news outlets or industry newsletters  
- Official websites, press releases, or blogs of the organizations involved  
- Scholarly databases (e.g., arXiv, PubMed, IEEE Xplore) for recent research  
- Real‑time data APIs if you’re looking for up‑to‑the‑minute statistics

Feel free to ask me anything within the scope of my knowledge up to

## Add web search tool

In [ ]:
import tavily
import typing
from langchain import tools

tavily_client = tavily.TavilyClient()

@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:
    """
    Search the web for information
    """

    return tavily_client.search(query=query)

web_search.invoke(input="""
    List of public holidays in Malaysia for 2026.
""")

{'query': 'List of public holidays in Malaysia for 2026.',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://sg.trip.com/guide/info/malaysia-public-holiday-2026.html',
   'title': 'Malaysia Public Holidays 2026: Calendar & Travel Guide',
   'content': '| ****Mar 20–22\\***** | Hari Raya Aidilfitri | 2 days (min) | End of Ramadan with open houses and festive meals | Peak travel, closures, early bookings needed |. Stay on top of your travel plans with our Malaysia Public Holiday 2026 calendar. Celebrations like ****Chinese New Year, Hari Raya Aidilfitri, and Merdeka Day**** often mean heavy traffic, packed malls, crowded tourist spots, and hotels that get snapped up fast in places like Kuala Lumpur, Penang, Langkawi, and Genting Highlands. Planning a trip to Malaysia during public holidays can be exciting because you get to see the country’s lively festivals up close. Some travellers love the energy of festive seasons like Chinese New Year and Ha

In [ ]:
import os
import time
import tavily
import typing
from langchain import chat_models, tools, agents, messages

tavily_client = tavily.TavilyClient()

@tools.tool
def web_search(query: str) -> typing.Dict[str, typing.Any]:
    """
    Search the web for information
    """

    return tavily_client.search(query=query)

# Initialize using the OpenAI provider with a custom base_url
llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

system_prompt = """
    You are a helpful assistant.
    Please keep the below structure.

    Country code: The country code of the holiday.
    Date: The date of the holiday.
    Name: The name of the holiday.
"""

agent = agents.create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt
)

question = messages.HumanMessage(content="""
    List of public holidays in Malaysia for 2026.
""")

start_time = time.time()
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 33.19s seconds
================================ Human Message =================================


    List of public holidays in Malaysia for 2026.

================================== Ai Message ==================================
Tool Calls:
  web_search (call_jg2g39gy)
 Call ID: call_jg2g39gy
  Args:
    query: Malaysia public holidays 2026
================================= Tool Message =================================
Name: web_search

{"query": "Malaysia public holidays 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://calendarific.com/holidays/2026/MY", "title": "Malaysia Public Holidays in 2026", "content": "| **Federal Territory Day** | Sunday, February 1, 2026 |. | **Federal Territory Day observed** | Monday, February 2, 2026 |. | **Day off for Thaipusam** | Monday, February 2, 2026 |. | **Valentine's Day** | Saturday, February 14, 2026 |. | **Chinese New Year's Day** | Tuesday, February 17, 2026 |. | **Second Day of Chin